---
title: "2026 Final Exam, forecasting problem"
subtitle: "Machine Learning"
date: "May 2026"
---

<h1 style="color:red">
Fill your name here
</h1>

### The total grading for this part is 4 points.

<h2 style="color:red;">Docker container instructions.</h2>

<ul style="color: blue">
  <li>After the git pull, the folder called final26 should have been created, containing this notebook.</li>

  <li>Pay attention to the folder from where you started the container, you may need to do a cd (there is a cell for that below). 
  <div style="font-weight:bold">DO NOT USE ABSOLUTE PATHS!</div></li>
</ul>



<h2 style="color:red;">Exam instructions: read carefully!</h2>

#### Using this notebook

- **[Use this Jupyter notebook]{.underline}** to complete the required tasks. Keep the sectioning structure of the notebook and insert the code cells you need in the corresponding sections.Be mindful of the format and sectioning of your work, as it will be considered in grading.

- The notebook should contain the code with your **analysis** and **it must be reproducible**. Set the random seeds to ensure that.

- The **most important part of your work is the comments and interpretation** of the analysis results obtained. **Do not include uncommented figures**. Remember to include a **conclusion section** at the end. 


#### Submitting your work

Once you have finished:
+ **Make sure that your name appears on the top cells of the notebook.**
+ **Make sure to save this notebook.**
+ **Make sure that you are logged into your university account and use this link to upload the notebook**  
https://upcomillas-my.sharepoint.com/:f:/g/personal/fsansegundo_comillas_edu/IgADJSVTYjfNQp9Ox22-7BQ-AaomoPGeDPKoLA3L5yGnJ4s

---

::: {.callout-warning icon=false}

##### Setting the working directory

We begin by using cd to make sure we are in the right folder.

:::

In [2]:
!pwd

/wd


In [ ]:
%cd CHANGEME_TO_YOUR_PATH
!ls

# Setup

### Load Libraries



In [5]:
# System and Environment Setup

import os
import sys
import random
from tqdm import tqdm
from typing import Optional

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")


# warnings.filterwarnings(
#     "ignore", 
#     category=UserWarning, 
#     message=".*FigureCanvasAgg is non-interactive.*")



# Data Science Stack
import numpy as np
import pandas as pd
pd.set_option("max_colwidth", 100)
pd.set_option("display.precision", 3)
import scipy.stats as stats
from sklearn.metrics import mean_squared_error, mean_absolute_error


# Plotting and Plot Configuration
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
%config InlineBackend.figure_format = 'png' 
from matplotlib.ticker import MaxNLocator
from IPython.display import Image
sns.set_context('notebook')
from cycler import cycler
# Custom color cycle 
# mpl.rcParams['axes.prop_cycle'] = cycler(color=["#000000", "#000000"])
# Style and Aesthetics
plt.style.use("ggplot")
sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "figure.constrained_layout.use": True,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "legend.title_fontsize": 10,
})


# Forecasting
import datetime
import yfinance as yf
import rdatasets
import pyreadr

import statsmodels.api as sm
import statsmodels.stats.api as sms
import statsmodels.api as sm
import statsmodels.tsa as tsa
from statsmodels.formula.api import ols
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose, STL
# from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.tsa.arima.model import ARIMA as smARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
# from statsmodels.stats.diagnostic import acorr_ljungbox


import pmdarima as pmd

# from sktime.datasets import load_airline
# from sktime.utils.plotting import plot_series as plot_series_sktime
from sklearn.model_selection import TimeSeriesSplit

## Nixtla's Related Forecasting Libraries
# from fpppy.utils import plot_series as plot_series_fpp
from utilsforecast.plotting import plot_series as plot_series_utils
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import rmse, mape, mae, smape
from statsforecast import StatsForecast
from statsforecast.arima import ARIMASummary, ndiffs, nsdiffs
from statsforecast.models import SeasonalNaive, ARIMA, AutoARIMA, AutoETS
os.environ["NIXTLA_ID_AS_COL"] = "true"

# Reproducibility
np.random.seed(1)
random.seed(1)
np.set_printoptions(suppress=True)

### Auxiliary function

This function plots a timeplot of the series, its ACF and PACF. You can change the lags parameter.

In [6]:
def plot_acf_pacf(df_ts, var, title="Time Series", lags = 10, plot_points=False):

    fig, axs = plt.subplots(3, 1, figsize=(8, 9), sharex=False, sharey=False)

    # Plot Time Series
    ax0 = axs[0]
    df_ts[var].plot(color="blue", ax=ax0)
    if plot_points:
	    pd.DataFrame({"t":range(df_ts.shape[0]), 
	    	var:df_ts[var]}).plot(color="blue", kind="scatter", x="t", y=var, ax=ax0)
    ax0.set_title(title)
    ax0.grid(visible=True, which='both', axis='x')
    ax0.grid(visible=False, which='Major', axis='y')
    # Plot ACF
    ax1 = axs[1]
    sm.graphics.tsa.plot_acf(df_ts[var].dropna(), ax=ax1, lags=lags, zero=False, title='ACF')
    ax1.set(ylim=(-1,1), xlabel='Lag', ylabel='acf')
    # Plot PACF
    ax2 = axs[2]
    sm.graphics.tsa.plot_pacf(df_ts[var].dropna(), ax=ax2, lags=lags, zero=False, title='PACF')
    ax2.set(ylim=(-1,1), xlabel='Lag', ylabel='pacf')

    plt.tight_layout()
    plt.show();plt.close()

---

::: {.callout-note icon="false}

# **STATEMENT**

Daily Flight Information Dataset

The data is provided in the file 

`DailyFlightInfo.csv`

It contains a monthly time series with information about airline passengers in the US. The meaning of the variables is as follows:

+ The `Date` of the observation.
+ The total count of `Passengers`.
+ The total count of `Flights`.
+ `Revenue` is a metric that represents one paying passenger flown one mile
+ `Capacity` indicates the airlines total passenger-carrying capacity, whether occupied or empty.

:::

::: {.callout-tip icon="false}

### Question 1:  Exploratory Data Analysis (EDA) and data preparation (1 point). 

- Load the dataset using pandas. 
- Perform data exploration and thorough cleaning of the dataset.
- Visualize the time series. Explore the trend and seasonality or seasonalities of this time series.
- Preprocess the data as needed. 
- **VERY IMPORTANT:** The dataset covers a time period from January 2003 to September 2023. Split the data into training and testing sets. 
    + **The training dataset is from 2003-01 to 2018-12 (both included).**
    + **The test set is 12 months, from 2019-01 to 2019-12 (both included)**. 
    + In particular, the last segment of the series **from 2020-01 to 2023-09 is ONLY used in an optional question**.
+ For validation in the training set, use a **forecasting horizon of 12 months**, and a step size to get **CV with 10 folds**.
:::

In [ ]:
df = pd.read_csv("DailyFlightInfo.csv")
df.head()


::: {.callout-tip icon="false}

### Question 2 : Identification and Fitting Process of a *seasonal last* naive model as baseline (1 point)

+ Fit **a seasonal last naive** model to the data.
+ Evaluate the performance of the model using RMSE and MAPE in the test set and in CV. This will be the performance metrics used for the selection between the models. 
+ Visualize the predictions of this baseline model.


:::


::: {.callout-tip icon="false}

### Question 3: SARIMA model (1 point)

- Select and diagnose a SARIMA model for the time series.
- Evaluate the performance of this model in validation and test.
- Visualize the predictions of this model.

:::


::: {.callout-tip icon="false}

### Question 4: SARIMAX model (1 point)

- Select and diagnose a SARIMAX model using `Flights`, `Revenue` and `Capacity` as exogenous variables.
- Evaluate the performance of this model in validation and test with the same metrics you used for previous models.
- **VERY IMPORTANT:** When evaluating the model performance, assume that the future values of the exogenous variables are known. 
- Visualize the predictions of this model.

:::


::: {.callout-tip icon="false}

### Question 4: Final model comparison and conclusions (points)

- Summarize your findings from the comparative analysis of these forecasting models. 

:::


::: {.callout-tip icon="false}

###  Optional 1: 

Discuss (without actually doing it) what you would change if the training period ends in 2022-09 and the test period is up to 2023-09.

###  Optional 2: 

Discuss (without actually doing it) what you would change if the future values of the exogenous variables are not known. How do you expect this to impact in the results of the SARIMAX model?

:::